# Sunrise/Sunset Analytics

This notebook connects to the local SQLite database populated by the ETL pipeline
and answers the required analytical questions:

- Longest and shortest day per calendar year
- Latest sunrise time per month
- Earliest sunset time per month

Run the pipeline first so that `output/sunrise_sunset.db` exists and is populated.


In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

DB_PATH = Path("output") / "sunrise_sunset.db"
DB_PATH

WindowsPath('output/sunrise_sunset.db')

In [2]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"Database not found at {DB_PATH}. Run the ETL pipeline first.")

conn = sqlite3.connect(DB_PATH)

# Load all records into a DataFrame
df = pd.read_sql("SELECT date, day_length, sunrise, sunset FROM sun_events", conn)

# Parse dates/times
df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
df["sunrise"] = pd.to_datetime(df["sunrise"], errors="coerce")
df["sunset"] = pd.to_datetime(df["sunset"], errors="coerce")

df.head()

,date,day_length,sunrise,sunset
0,2020-01-01,None,2020-01-01 07:12:26+00:00,2020-01-01 17:36:20+00:00
1,2020-01-02,None,2020-01-02 07:12:41+00:00,2020-01-02 17:37:01+00:00
2,2020-01-03,None,2020-01-03 07:12:55+00:00,2020-01-03 17:37:43+00:00
3,2020-01-04,None,2020-01-04 07:13:08+00:00,2020-01-04 17:38:26+00:00
4,2020-01-05,None,2020-01-05 07:13:18+00:00,2020-01-05 17:39:10+00:00


## Longest and shortest day per calendar year


In [3]:
# Add year column
df["year"] = df["date"].dt.year

# Filter rows where day_length is present
df_valid = df.dropna(subset=["day_length"]).copy()

# Compute day length in seconds reliably
# If the `day_length` column is populated, use it; otherwise derive it
# from sunrise/sunset timestamps.

if "day_length" in df.columns and df["day_length"].notna().any():
    df["day_length_sec"] = df["day_length"]
else:
    df["day_length_sec"] = (df["sunset"] - df["sunrise"]).dt.total_seconds()

# Add year column
df["year"] = df["date"].dt.year

# Drop rows without a valid day length
df_valid = df.dropna(subset=["day_length_sec"]).copy()

# Longest day per year
idx_longest = df_valid.groupby("year")["day_length_sec"].idxmax()
longest_days = df_valid.loc[idx_longest, ["year", "date", "day_length_sec"]].sort_values("year")

# Shortest day per year
idx_shortest = df_valid.groupby("year")["day_length_sec"].idxmin()
shortest_days = df_valid.loc[idx_shortest, ["year", "date", "day_length_sec"]].sort_values("year")

print("Longest day per year:")
display(longest_days)

print("\nShortest day per year:")
display(shortest_days)

Longest day per year:


,year,date,day_length_sec
171,2020,2020-06-20,50440.0
537,2021,2021-06-21,50440.0
902,2022,2022-06-21,50440.0
1267,2023,2023-06-21,50440.0
1632,2024,2024-06-20,50440.0
1998,2025,2025-06-21,50440.0
2254,2026,2026-03-04,42122.0



Shortest day per year:


,year,date,day_length_sec
355,2020,2020-12-21,37309.0
720,2021,2021-12-21,37310.0
1085,2022,2022-12-21,37309.0
1450,2023,2023-12-21,37310.0
1816,2024,2024-12-21,37309.0
2181,2025,2025-12-21,37310.0
2192,2026,2026-01-01,37449.0


## Latest sunrise and earliest sunset per month


In [3]:
# Define a year-month key for grouping
df["year_month"] = df["date"].dt.to_period("M")

# Latest sunrise per month (max sunrise time)
df_sunrise = df.dropna(subset=["sunrise"]).copy()
idx_latest_sunrise = df_sunrise.groupby("year_month")["sunrise"].idxmax()
latest_sunrise = df_sunrise.loc[idx_latest_sunrise, ["year_month", "date", "sunrise"]].sort_values("year_month")

# Earliest sunset per month (min sunset time)
df_sunset = df.dropna(subset=["sunset"]).copy()
idx_earliest_sunset = df_sunset.groupby("year_month")["sunset"].idxmin()
earliest_sunset = df_sunset.loc[idx_earliest_sunset, ["year_month", "date", "sunset"]].sort_values("year_month")

print("Latest sunrise per month:")
display(latest_sunrise)

print("\nEarliest sunset per month:")
display(earliest_sunset)

Latest sunrise per month:


,year_month,date,sunrise
30,2020-01,2020-01-31,2020-01-31 07:08:46+00:00
59,2020-02,2020-02-29,2020-02-29 06:45:35+00:00
90,2020-03,2020-03-31,2020-03-31 06:10:40+00:00
120,2020-04,2020-04-30,2020-04-30 05:39:38+00:00
151,2020-05,2020-05-31,2020-05-31 05:22:27+00:00
...,...,...,...
2160,2025-11,2025-11-30,2025-11-30 06:54:25+00:00
2191,2025-12,2025-12-31,2025-12-31 07:12:17+00:00
2222,2026-01,2026-01-31,2026-01-31 07:08:28+00:00
2250,2026-02,2026-02-28,2026-02-28 06:46:03+00:00



Earliest sunset per month:


,year_month,date,sunset
0,2020-01,2020-01-01,2020-01-01 17:36:20+00:00
31,2020-02,2020-02-01,2020-02-01 18:01:00+00:00
60,2020-03,2020-03-01,2020-03-01 18:22:17+00:00
91,2020-04,2020-04-01,2020-04-01 18:40:22+00:00
121,2020-05,2020-05-01,2020-05-01 18:57:40+00:00
...,...,...,...
2131,2025-11,2025-11-01,2025-11-01 17:37:27+00:00
2161,2025-12,2025-12-01,2025-12-01 17:25:11+00:00
2192,2026-01,2026-01-01,2026-01-01 17:36:42+00:00
2223,2026-02,2026-02-01,2026-02-01 18:01:26+00:00
